In [1]:
import os

root_dir = "/kaggle/input/datasets/afqwe0/clotheslabelled"
total_list = []
for dirpath, dirnames, filenames in os.walk(root_dir):
    for filename in filenames:
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            full_path = os.path.join(dirpath, filename)
        total_list.append(full_path)
print(total_list[0])

/kaggle/input/datasets/afqwe0/clotheslabelled/Himesh/fit-and-flare dress/001797.jpg


In [2]:
from pathlib import Path
Path('/kaggle/working/CSV_files').mkdir(parents=True, exist_ok=True)

In [3]:
import os
import pandas as pd

label_dir = "/kaggle/input/datasets/afqwe0/clotheslabelled/Labels"

rows = []

for image_path in total_list:

    # Get corresponding label filename
    image_name = os.path.splitext(os.path.basename(image_path))[0]
    label_path = os.path.join(label_dir, image_name + ".txt")

    # Skip if label doesn't exist
    if not os.path.exists(label_path):
        continue

    with open(label_path, "r") as f:
        for line in f:

            line = line.strip()
            if not line:
                continue

            class_id, x_center, y_center, width, height = line.split()

            rows.append({
                "image_path": image_path,
                "class_id": int(class_id),
                "x_center": float(x_center),
                "y_center": float(y_center),
                "width": float(width),
                "height": float(height)
            })

# Create DataFrame
df = pd.DataFrame(rows)

# Save CSV
df.to_csv("/kaggle/working/CSV_files/annotations.csv", index=False)

print(df.head())
print(f"\nSaved {len(df)} annotations.")

                                          image_path  class_id  x_center  \
0  /kaggle/input/datasets/afqwe0/clotheslabelled/...         7  0.507507   
1  /kaggle/input/datasets/afqwe0/clotheslabelled/...         7  0.505364   
2  /kaggle/input/datasets/afqwe0/clotheslabelled/...         7  0.419720   
3  /kaggle/input/datasets/afqwe0/clotheslabelled/...         7  0.418930   
4  /kaggle/input/datasets/afqwe0/clotheslabelled/...         7  0.440800   

   y_center     width    height  
0  0.617903  0.554344  0.669839  
1  0.616825  0.554344  0.666374  
2  0.679029  0.383289  0.497177  
3  0.679118  0.384946  0.501083  
4  0.608129  0.439121  0.683511  

Saved 7795 annotations.


In [19]:
Path('/kaggle/working/Cropped_Images').mkdir(parents=True, exist_ok=True)

In [20]:
CLASS_NAMES = {
    0: "t_shirt",
    1: "trousers",
    2: "blazer",
    3: "hoodie",
    4: "sneakers",
    5: "dolphin_shorts",
    6: "denim_shorts",
    7: "fit_and_flare_dress",
    8: "jacket",
    9: "skirt",
    10: "men_suits_western_coatpant",
    11: "loafers",
    12: "formal_pants_shirt_set",
    13: "dhaka_topi",
    14: "sari",
    15: "baggy_pants",
    16: "jeans_pants",
    17: "one_piece",
    18: "sweatshirt",
    19: "caps",
    20: "cardigan",
    21: "cowl_neck_top",
    22: "turtleneck",
    23: "polo",
    24: "sweater",
}

In [21]:
import os
import pandas as pd
from PIL import Image

df = pd.read_csv("/kaggle/working/CSV_files/annotations.csv")

output_dir = "/kaggle/working/Cropped_Images"
os.makedirs(output_dir, exist_ok=True)

crop_counts = {}

for _, row in df.iterrows():

    image_path = row["image_path"]

    image = Image.open(image_path)
    img_w, img_h = image.size

    x_center = row["x_center"] * img_w
    y_center = row["y_center"] * img_h
    width = row["width"] * img_w
    height = row["height"] * img_h

    x1 = int(x_center - width / 2)
    y1 = int(y_center - height / 2)
    x2 = int(x_center + width / 2)
    y2 = int(y_center + height / 2)

    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(img_w, x2)
    y2 = min(img_h, y2)

    crop = image.crop((x1, y1, x2, y2))

    filename = os.path.splitext(os.path.basename(image_path))[0]

    i = crop_counts.get(image_path, 0)

    save_name = f"{CLASS_NAMES[row['class_id']]}_{filename}_crop_{i}.png"

    crop.save(os.path.join(output_dir, save_name))

    crop_counts[image_path] = i + 1

print("Done!")

Done!
